# NB 01 — Extracción de Facturas
**Proyecto 5 · Automatización Contable**

**Input:** PDFs e imágenes en `data/input/compras/` y `data/input/ventas/`  
**Output:** JSON estructurado por factura en `data/processed/`

**Empresa cliente:** CT PRIME CONSULTING SAC · RUC 20563642930

## 0. Dependencias

In [1]:
import anthropic
import base64
import json
import fitz  # pymupdf
import io
import os
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

# Cargar API key desde .env compartido
ENV_PATH = Path('D:/Proyecto_Gabriel/02_Agente_IA/Skill_financiero/.env')
load_dotenv(dotenv_path=ENV_PATH)

BASE_DIR      = Path('../')
INPUT_COMPRAS = BASE_DIR / 'data/input/compras'
INPUT_VENTAS  = BASE_DIR / 'data/input/ventas'
OUTPUT_DIR    = BASE_DIR / 'data/processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUC_EMPRESA = '20563642930'

client = anthropic.Anthropic()
api_ok = 'SI' if os.environ.get('ANTHROPIC_API_KEY') else 'NO ENCONTRADA'
print(f'Setup OK - PyMuPDF {fitz.version[0]}')
print(f'API key cargada: {api_ok}')


Setup OK - PyMuPDF 1.27.2.3
API key cargada: SI


## 0b. Copia inicial de facturas de referencia
Ejecutar solo la primera vez para copiar las facturas desde `Proyecto_contable/`.
En producción, Carlos coloca los PDFs directamente en `data/input/compras/` o `data/input/ventas/`.

In [2]:
import shutil

# Rutas de las facturas de referencia
SRC_COMPRAS = Path('D:/Proyecto_Gabriel/01_Portfolio/Proyecto_contable/facturas_proveedores')
SRC_VENTAS  = Path('D:/Proyecto_Gabriel/01_Portfolio/Proyecto_contable/factuas_clientes')

def copiar_facturas_referencia():
    copiadas = 0
    for src, dst in [(SRC_COMPRAS, INPUT_COMPRAS), (SRC_VENTAS, INPUT_VENTAS)]:
        if not src.exists():
            print(f'  Carpeta no encontrada: {src}')
            continue
        for archivo in src.iterdir():
            if archivo.suffix.lower() in {'.pdf', '.jpg', '.jpeg', '.png'}:
                dest = dst / archivo.name
                if not dest.exists():  # no sobreescribir si ya fue copiado
                    shutil.copy2(archivo, dest)
                    copiadas += 1
    print(f'Facturas copiadas: {copiadas}')
    print(f'  Compras: {len(list(INPUT_COMPRAS.iterdir()))} archivos')
    print(f'  Ventas:  {len(list(INPUT_VENTAS.iterdir()))} archivos')

copiar_facturas_referencia()

Facturas copiadas: 0
  Compras: 0 archivos
  Ventas:  3 archivos


## 1. Funciones de Conversión (PDF / Imagen → base64)

In [3]:
def archivo_a_base64(ruta: Path) -> tuple[str, str]:
    """
    Convierte PDF o imagen a base64 usando PyMuPDF (no requiere Poppler).
    Para PDFs renderiza la primera página a JPEG 200 DPI.
    """
    ext = ruta.suffix.lower()

    if ext == '.pdf':
        doc = fitz.open(str(ruta))
        page = doc[0]  # primera página
        mat = fitz.Matrix(200/72, 200/72)  # 200 DPI
        pix = page.get_pixmap(matrix=mat, alpha=False)
        img_bytes = pix.tobytes('jpeg')
        doc.close()
        return base64.b64encode(img_bytes).decode(), 'image/jpeg'

    elif ext in ['.jpg', '.jpeg']:
        with open(ruta, 'rb') as f:
            return base64.b64encode(f.read()).decode(), 'image/jpeg'

    elif ext == '.png':
        with open(ruta, 'rb') as f:
            return base64.b64encode(f.read()).decode(), 'image/png'

    else:
        raise ValueError(f'Formato no soportado: {ext}')

## 2. Extracción con Claude API (visión)

In [4]:
PROMPT_EXTRACCION = """
Eres un asistente especializado en documentos tributarios peruanos.
Extrae TODOS los campos del comprobante de pago en la imagen.

Devuelve ÚNICAMENTE un JSON con esta estructura exacta (sin texto adicional):

{
  "tipo_doc": "FACTURA | BOLETA | NOTA_CREDITO | NOTA_DEBITO | RECIBO",
  "codigo_tipo_doc": "01 | 03 | 07 | 08 | R1 | 04",
  "serie": "F001",
  "numero": "00012345",
  "fecha_emision": "YYYY-MM-DD",
  "fecha_vencimiento_pago": "YYYY-MM-DD o null",
  "ruc_emisor": "20XXXXXXXXX",
  "razon_social_emisor": "Nombre legal del emisor",
  "ruc_receptor": "20XXXXXXXXX o null",
  "razon_social_receptor": "Nombre legal del receptor o null",
  "tipo_doc_identidad_receptor": "1 | 4 | 6 | 7 | 0",
  "numero_doc_identidad_receptor": "numero de DNI/RUC/pasaporte del receptor, o null",
  "descripcion_servicio": "Descripción principal del bien/servicio",
  "base_imponible": 0.00,
  "igv": 0.00,
  "total": 0.00,
  "moneda": "PEN | USD",
  "tipo_cambio": 0.000,
  "tiene_detraccion": false,
  "monto_detraccion": 0.00,
  "doc_referencia": "E001-00000030 o null",
  "otorga_credito_igv": true,
  "deducible_renta": true,
  "confianza_extraccion": 0.95
}

Reglas:
- codigo_tipo_doc: 01=Factura, 03=Boleta, 07=Nota de Crédito, 08=Nota de Débito, R1=Recibo por Honorarios, 04=Liquidación de Compra
- razon_social_emisor y razon_social_receptor: deben ser la denominación o razón social LEGAL de la empresa (ej: "MEDIA SOLUTION E.I.R.L.", "SODIMAC PERU S.A."). NUNCA una dirección, domicilio fiscal, ciudad ni referencia geográfica. Si el documento muestra el domicilio debajo del nombre de la empresa, extrae SOLO el nombre legal.
- Para NC/ND: "numero" debe ser el número PROPIO de la NC/ND (ej: para "NC E001-03", numero="00000003"), NO el número del comprobante que modifica. El comprobante modificado va ÚNICAMENTE en "doc_referencia".
- doc_referencia: para tipo 07 (NC) y 08 (ND), extraer la serie-número del comprobante que modifica (ej: "E001-00000030"). Para otros tipos, null.
- otorga_credito_igv: true si codigo_tipo_doc es 01 o 04 y el IGV está discriminado. false para 03 (Boleta) y R1 (Honorarios).
- deducible_renta: true para 01 (Factura), R1 (Honorarios) y 04 (Liquidación Compra). false para 03 (Boleta) por regla general.
- Para NC/ND (07/08): otorga_credito_igv y deducible_renta heredan del comprobante que modifican — dejar true por defecto.
- Si base_imponible no aparece explícita, calcúlala: total / 1.18
- Si IGV no aparece, calcúlalo: base_imponible * 0.18
- fecha_vencimiento_pago: solo si el comprobante muestra explícitamente una fecha de vencimiento distinta a la de emisión; si no aparece, usa null.
- tipo_doc_identidad_receptor: 1=DNI, 4=Carnet de Extranjería, 6=RUC, 7=Pasaporte, 0=otro/no identificado. Para FACTURA siempre es 6. Para BOLETA, usa el documento que aparezca (DNI es lo más común); si no hay documento de identidad visible, usa "0".
- numero_doc_identidad_receptor: el número del documento indicado en tipo_doc_identidad_receptor. Si es RUC, debe coincidir con ruc_receptor.
- tipo_cambio: solo si moneda es "USD" y el comprobante muestra el tipo de cambio explícitamente; si no aparece o moneda es "PEN", usa null.
- confianza_extraccion: 1.0 = todos los campos legibles, 0.7 = imagen borrosa o campos incompletos
- Devuelve SOLO el JSON, sin markdown, sin explicaciones
"""

def extraer_factura(ruta: Path, tipo_operacion: str) -> dict:
    """
    tipo_operacion: 'COMPRA' o 'VENTA'
    Retorna dict con todos los campos extraídos + metadatos.
    """
    b64, media_type = archivo_a_base64(ruta)
    
    response = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=1024,
        messages=[{
            'role': 'user',
            'content': [
                {
                    'type': 'image',
                    'source': {'type': 'base64', 'media_type': media_type, 'data': b64}
                },
                {'type': 'text', 'text': PROMPT_EXTRACCION}
            ]
        }]
    )
    
    texto = response.content[0].text.strip()
    # Limpiar markdown si Claude lo devuelve con bloques ```json
    if texto.startswith('```'):
        texto = texto.split('```')[1]
        if texto.startswith('json'):
            texto = texto[4:]
    
    datos = json.loads(texto)
    datos['tipo_operacion'] = tipo_operacion
    datos['archivo_origen'] = ruta.name
    datos['procesado_en'] = datetime.now().isoformat()
    
    return datos

## 3. Procesamiento en lote

In [5]:
FORMATOS_VALIDOS = {'.pdf', '.jpg', '.jpeg', '.png'}

def procesar_carpeta(carpeta: Path, tipo_operacion: str) -> list[dict]:
    archivos = [f for f in carpeta.iterdir() if f.suffix.lower() in FORMATOS_VALIDOS]
    resultados = []
    errores = []
    
    print(f'\n=== {tipo_operacion}: {len(archivos)} archivos ===')
    
    for archivo in sorted(archivos):
        print(f'  Procesando: {archivo.name}... ', end='')
        try:
            datos = extraer_factura(archivo, tipo_operacion)
            resultados.append(datos)
            print(f'OK (confianza: {datos.get("confianza_extraccion", "?")})')
        except Exception as e:
            print(f'ERROR: {e}')
            errores.append({'archivo': archivo.name, 'error': str(e)})
    
    if errores:
        print(f'\nErrores ({len(errores)}):')
        for e in errores:
            print(f'  - {e["archivo"]}: {e["error"]}')
    
    return resultados

In [6]:
# Ejecutar extracción
facturas_compras = procesar_carpeta(INPUT_COMPRAS, 'COMPRA')
facturas_ventas  = procesar_carpeta(INPUT_VENTAS,  'VENTA')

todas_las_facturas = facturas_compras + facturas_ventas

# Deduplicación dentro del lote por (ruc_emisor, serie, numero)
vistas = set()
facturas_unicas = []
for f in todas_las_facturas:
    clave = (str(f.get('ruc_emisor', '')), str(f.get('serie', '')), str(f.get('numero', '')))
    if clave not in vistas:
        vistas.add(clave)
        facturas_unicas.append(f)
    else:
        print(f'  ⚠️ Duplicado en lote: RUC={clave[0]} {clave[1]}-{clave[2]} — descartado')
todas_las_facturas = facturas_unicas

n_compras = sum(1 for f in todas_las_facturas if f.get('tipo_operacion') == 'COMPRA')
n_ventas  = sum(1 for f in todas_las_facturas if f.get('tipo_operacion') == 'VENTA')
print(f'\nTotal único: {len(todas_las_facturas)} ({n_compras} compras, {n_ventas} ventas)')


=== COMPRA: 0 archivos ===

=== VENTA: 3 archivos ===
  Procesando: FACT E001-30 MEDIA SOLUTION.pdf... 

OK (confianza: 0.85)
  Procesando: FACT E001-31 MEDIA SOLUTION.pdf... 

OK (confianza: 0.95)
  Procesando: NC E001-03 ANULA FACT E001-30 MEDIA SOLUTION.pdf... 

OK (confianza: 0.92)

Total único: 3 (0 compras, 3 ventas)


## 4. Guardar resultados

In [7]:
# Guardar JSON individual por factura
for factura in todas_las_facturas:
    nombre_base = Path(factura['archivo_origen']).stem
    salida = OUTPUT_DIR / f'{nombre_base}_extraido.json'
    with open(salida, 'w', encoding='utf-8') as f:
        json.dump(factura, f, ensure_ascii=False, indent=2)

# Guardar resumen consolidado
resumen_path = OUTPUT_DIR / 'facturas_extraidas.json'
with open(resumen_path, 'w', encoding='utf-8') as f:
    json.dump(todas_las_facturas, f, ensure_ascii=False, indent=2)

print(f'Archivos guardados en: {OUTPUT_DIR}')
print(f'Resumen consolidado: {resumen_path}')

Archivos guardados en: ..\data\processed
Resumen consolidado: ..\data\processed\facturas_extraidas.json


## 5. Revisión de resultados

In [8]:
import pandas as pd

df = pd.DataFrame(todas_las_facturas)
cols_display = ['tipo_operacion', 'tipo_doc', 'serie', 'numero', 'fecha_emision',
                'razon_social_emisor', 'base_imponible', 'igv', 'total', 'confianza_extraccion']
cols_disponibles = [c for c in cols_display if c in df.columns]

print('=== FACTURAS EXTRAÍDAS ===')
print(df[cols_disponibles].to_string(index=False))

print(f'\n=== CONFIANZA BAJA (< 0.80) — revisar manualmente ===')
baja_confianza = df[df['confianza_extraccion'] < 0.80] if 'confianza_extraccion' in df.columns else pd.DataFrame()
if len(baja_confianza) > 0:
    print(baja_confianza[['archivo_origen', 'confianza_extraccion']].to_string(index=False))
else:
    print('  Ninguna — todas las facturas con confianza >= 0.80')

=== FACTURAS EXTRAÍDAS ===
tipo_operacion     tipo_doc serie   numero fecha_emision         razon_social_emisor  base_imponible     igv  total  confianza_extraccion
         VENTA      FACTURA  E001 00000030    2025-02-20 CONTACTO CREATIVO TI S.A.C.         6355.93 1494.07 8850.0                  0.85
         VENTA      FACTURA  E001 00000031    2025-02-25 CONTACTO CREATIVO TI S.A.C.         1500.00  270.00 1770.0                  0.95
         VENTA NOTA_CREDITO  E001 00000003    2025-02-25 CONTACTO CREATIVO TI S.A.C.         7500.00 1350.00 8850.0                  0.92

=== CONFIANZA BAJA (< 0.80) — revisar manualmente ===
  Ninguna — todas las facturas con confianza >= 0.80
